In [1]:
import torch
print(torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

MODEL = "/kaggle/input/models/aisuko/xlm-roberta-base/transformers/default/1"
DATA   = "/kaggle/input/competitions/rmit-hackathon-2026"

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I    = {l: i for i, l in enumerate(LABELS)}

MAXLEN, EPOCHS, LR = 256, 1, 2e-5
BS_PER_GPU = 16          # 2 x T4 -> effective train batch 32
NGPU = max(torch.cuda.device_count(), 1)

train = pd.read_csv(f"{DATA}/train.csv")[:200]
test  = pd.read_csv(f"{DATA}/test.csv")

tr, va = train_test_split(train, test_size=0.1, stratify=train.label, random_state=42)

tok = AutoTokenizer.from_pretrained(MODEL)

def mk(df, labelled=True):
    d = df[["text"]].copy()
    if labelled:
        d["labels"] = df.label.map(L2I).values
    return (Dataset.from_pandas(d, preserve_index=False)
            .map(lambda b: tok(b["text"], truncation=True, max_length=MAXLEN),
                 batched=True, remove_columns=["text"]))

ds_tr, ds_va, ds_te = mk(tr), mk(va), mk(test, labelled=False)
print(len(ds_tr), len(ds_va), len(ds_te))

ModuleNotFoundError: No module named 'torch'

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, 
    num_labels=len(LABELS),
    id2label={i: l for l, i in L2I.items()}, 
    label2id=L2I
)

print(model)

In [2]:
import kagglehub
kagglehub.login()  # Login to Kaggle Hub
# Download latest version
path = kagglehub.competition_download('rmit-hackathon-2026')

print("Path to competition files:", path)

100%|██████████| 4.05M/4.05M [00:01<00:00, 2.69MB/s]

Extracting files...


Path to competition files: C:\Users\rrana\.cache\kagglehub\competitions\rmit-hackathon-2026
